In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    FloatSlider, IntSlider, RadioButtons,
    HTML, HTMLMath, VBox, HBox, Layout
)
from IPython.display import display

# ============================================================
# ONE-DIMENSIONAL HEAT EQUATION ON A FINITE INTERVAL
#
# u_t = sigma u_xx
#
# u(0,t) = u(L,t) = 0
#
# u(x,0) = f(x)
#
# u(x,t) =
# sum C_n sin(n*pi*x/L)
# exp[-sigma(n*pi/L)^2 t]
# ============================================================

plt.ioff()

# ============================================================
# JUPYTER / BINDER DISPLAY SETTINGS
# ============================================================

display(HTML("""
<style>

.container {
    width:98% !important;
    max-width:none !important;
}

.output_area,
.output_subarea,
.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll {
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow:visible !important;
    resize:none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display:none !important;
}

.heat-title {
    font-family:Arial, sans-serif;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.heat-label {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
}

.heat-value {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

.heat-radio .widget-radio-box {
    display:flex !important;
    flex-direction:row !important;
    flex-wrap:nowrap !important;
    gap:18px !important;
}

.heat-radio .widget-radio-box label {
    margin:0 !important;
    white-space:nowrap !important;
}

.heat-radio > label {
    display:none !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1180px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="heat-title" style="margin-bottom:8px;">
One-Dimensional Heat Equation on a Finite Interval
</div>

<div style="margin-bottom:5px;">
The equation uₜ=σuₓₓ is solved on 0≤x≤L with homogeneous
boundary conditions u(0,t)=u(L,t)=0.
</div>

<div style="margin-bottom:5px;">
Separation of variables leads to spatial eigenfunctions
sin(nπx/L) and temporal factors
exp[−σ(nπ/L)²t].
</div>

<div style="margin-bottom:5px;">
The Fourier sine coefficients are calculated numerically from
the selected initial temperature distribution f(x). They are
not entered manually.
</div>

<div>
<b>This notebook:</b> shows how the initial temperature profile
is reconstructed from Fourier modes and how the higher spatial
harmonics decay faster as time increases.
</div>

</div>
""")

# ============================================================
# MATHEMATICAL RELATIONS
# ============================================================

solution_math = HTMLMath(
    value=(
        r'\('
        r'u(x,t)='
        r'\displaystyle\sum_{n=1}^{\infty}'
        r'C_n\sin\left(\frac{n\pi x}{L}\right)'
        r'e^{-\sigma(n\pi/L)^2t}'
        r'\)'
    )
)

coefficient_math = HTMLMath(
    value=(
        r'\('
        r'C_n='
        r'\frac{2}{L}'
        r'\displaystyle\int_0^L'
        r'f(x)\sin\left(\frac{n\pi x}{L}\right)dx'
        r'\)'
    )
)

relations = VBox(
    [
        solution_math,
        coefficient_math
    ],
    layout=Layout(
        width='1050px',
        gap='3px'
    )
)

# ============================================================
# FIXED PARAMETERS
# ============================================================

L = 1.0
MAX_MODES = 40
NX = 700

x = np.linspace(
    0.0,
    L,
    NX
)

mode_numbers = np.arange(
    1,
    MAX_MODES + 1
)

# ============================================================
# INITIAL CONDITION SELECTOR
# ============================================================

initial_selector = RadioButtons(
    options=[
        ('Triangular', 'triangle'),
        ('Parabolic', 'parabolic'),
        ('Sine mixture', 'sines')
    ],
    value='triangle',
    description='',
    layout=Layout(
        width='500px'
    )
)

initial_selector.add_class(
    'heat-radio'
)

# ============================================================
# CONTROLS
# ============================================================

slider_style = {
    'description_width': '0px'
}

time_slider = FloatSlider(
    min=0.0,
    max=0.25,
    step=0.0025,
    value=0.0,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=Layout(
        width='250px'
    )
)

sigma_slider = FloatSlider(
    min=0.05,
    max=1.00,
    step=0.05,
    value=0.20,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=Layout(
        width='250px'
    )
)

modes_slider = IntSlider(
    min=1,
    max=MAX_MODES,
    step=1,
    value=12,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=Layout(
        width='250px'
    )
)

time_value = HTML(
    '<div class="heat-value">0.000</div>'
)

sigma_value = HTML(
    '<div class="heat-value">0.20</div>'
)

modes_value = HTML(
    '<div class="heat-value">12</div>'
)

def make_row(label, slider, value):

    return HBox(
        [
            HTML(
                f'<div class="heat-label">{label}</div>',
                layout=Layout(
                    width='125px',
                    min_width='125px'
                )
            ),

            slider,

            value
        ],
        layout=Layout(
            width='470px',
            height='38px',
            align_items='center'
        )
    )

controls_panel = VBox(
    [
        HTML("""
        <div class="heat-title" style="margin-bottom:7px;">
            Parameters
        </div>
        """),

        initial_selector,

        make_row(
            'Time t:',
            time_slider,
            time_value
        ),

        make_row(
            'Diffusivity σ:',
            sigma_slider,
            sigma_value
        ),

        make_row(
            'Modes N:',
            modes_slider,
            modes_value
        )
    ],
    layout=Layout(
        width='500px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

# ============================================================
# INFORMATION PANEL
# ============================================================

info_math = HTMLMath()

info_panel = VBox(
    [
        HTML("""
        <div class="heat-title" style="margin-bottom:7px;">
            Current Solution
        </div>
        """),

        info_math
    ],
    layout=Layout(
        width='620px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

top_row = HBox(
    [
        controls_panel,
        info_panel
    ],
    layout=Layout(
        width='1140px',
        gap='15px',
        align_items='stretch'
    )
)

# ============================================================
# INITIAL CONDITIONS
# ============================================================

def initial_function(mode):

    if mode == 'triangle':

        return (
            1.0
            -
            np.abs(
                2.0*x/L
                -
                1.0
            )
        )

    if mode == 'parabolic':

        return (
            4.0
            *
            x
            *
            (L-x)
            /
            (L**2)
        )

    return (
        np.sin(
            np.pi*x/L
        )
        +
        0.50
        *
        np.sin(
            3.0*np.pi*x/L
        )
        +
        0.25
        *
        np.sin(
            5.0*np.pi*x/L
        )
    )

# ============================================================
# FOURIER COEFFICIENTS
# ============================================================

def calculate_coefficients(
    f_values
):

    coeffs = np.zeros(
        MAX_MODES
    )

    for i, n in enumerate(
        mode_numbers
    ):

        integrand = (
            f_values
            *
            np.sin(
                n*np.pi*x/L
            )
        )

        coeffs[i] = (
            (2.0/L)
            *
            np.trapezoid(
                integrand,
                x
            )
        )

    return coeffs

# ============================================================
# INITIAL DATA
# ============================================================

f_initial = initial_function(
    initial_selector.value
)

coefficients = calculate_coefficients(
    f_initial
)

def build_solution(
    t,
    sigma,
    number_modes
):

    n = mode_numbers[
        :number_modes
    ]

    basis = np.sin(
        n[:, None]
        *
        np.pi
        *
        x[None, :]
        /
        L
    )

    decay = np.exp(
        -sigma
        *
        (n*np.pi/L)**2
        *
        t
    )

    return np.sum(
        coefficients[:number_modes, None]
        *
        basis
        *
        decay[:, None],
        axis=0
    )

u_initial = build_solution(
    time_slider.value,
    sigma_slider.value,
    modes_slider.value
)

# ============================================================
# FIGURE 1
# TEMPERATURE DISTRIBUTION
# ============================================================

fig_profile, ax_profile = plt.subplots(
    figsize=(7.0, 4.6)
)

fig_profile.canvas.header_visible = False
fig_profile.canvas.footer_visible = False
fig_profile.canvas.toolbar_visible = False

fig_profile.canvas.layout = Layout(
    width='700px',
    height='460px'
)

ax_profile.set_title(
    'Temperature Distribution',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax_profile.set_xlabel(
    'Position x'
)

ax_profile.set_ylabel(
    'Temperature'
)

ax_profile.set_xlim(
    0.0,
    L
)

ax_profile.set_ylim(
    -0.20,
    1.30
)

ax_profile.grid(
    True,
    linestyle=':',
    alpha=0.40
)

initial_line, = ax_profile.plot(
    x,
    f_initial,
    linewidth=1.7,
    linestyle='--',
    label='Initial f(x)'
)

solution_line, = ax_profile.plot(
    x,
    u_initial,
    linewidth=2.2,
    label='u(x,t)'
)

ax_profile.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=2,
    fontsize=9
)

fig_profile.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.90,
    bottom=0.23
)

# ============================================================
# FIGURE 2
# MODAL AMPLITUDES — STEM REPRESENTATION
# ============================================================

fig_modes, ax_modes = plt.subplots(
    figsize=(4.3, 4.6)
)

fig_modes.canvas.header_visible = False
fig_modes.canvas.footer_visible = False
fig_modes.canvas.toolbar_visible = False

fig_modes.canvas.layout = Layout(
    width='430px',
    height='460px'
)

ax_modes.set_title(
    'Modal Amplitudes',
    fontsize=13,
    fontweight='bold',
    color='#0b3d91'
)

ax_modes.set_xlabel(
    'Mode n'
)

ax_modes.set_ylabel(
    '|Cₙ exp(−λₙt)|'
)

ax_modes.set_xlim(
    0.5,
    modes_slider.value + 0.5
)

ax_modes.set_ylim(
    0.0,
    1.10
)

ax_modes.grid(
    True,
    axis='y',
    linestyle=':',
    alpha=0.40
)

# ------------------------------------------------------------
# Horizontal baseline
# ------------------------------------------------------------

ax_modes.axhline(
    0.0,
    linewidth=1.0
)

# ------------------------------------------------------------
# Stem vertical lines
#
# They are created ONCE.
# set_segments() is used during updates.
# ------------------------------------------------------------

initial_segments = [
    [
        (n, 0.0),
        (n, 0.0)
    ]
    for n in mode_numbers
]

stem_lines = ax_modes.vlines(
    mode_numbers,
    np.zeros(MAX_MODES),
    np.zeros(MAX_MODES),
    linewidth=1.2
)

# ------------------------------------------------------------
# Stem markers
# ------------------------------------------------------------

modal_points, = ax_modes.plot(
    mode_numbers,
    np.zeros(MAX_MODES),
    linestyle='None',
    marker='o',
    markersize=5
)

fig_modes.subplots_adjust(
    left=0.15,
    right=0.97,
    top=0.90,
    bottom=0.13
)

# ============================================================
# FIGURES ROW
# ============================================================

figures_row = HBox(
    [
        fig_profile.canvas,
        fig_modes.canvas
    ],
    layout=Layout(
        width='1145px',
        gap='10px',
        align_items='flex-start'
    )
)

# ============================================================
# CONCLUSION PANEL
# ============================================================

conclusion_panel = HTML("""
<div style="
    width:1135px;
    padding:10px 13px;
    border:1px solid #d7c7e5;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.55;
    box-sizing:border-box;
    margin-top:4px;
">

<div style="
    color:#6f3fa0;
    font-size:17px;
    font-weight:bold;
    margin-bottom:6px;
">
Interpretation
</div>

<div style="margin-bottom:6px;">
At <b>t = 0</b>, the solution satisfies the initial condition
<b>u(x,0) = f(x)</b>. Therefore, the solution curve coincides with
the initial temperature distribution, apart from the small numerical
difference that may result from retaining only a finite number of
Fourier modes.
</div>

<div style="margin-bottom:6px;">
As time increases, every spatial mode is multiplied by the factor
<b>exp[−σ(nπ/L)²t]</b>. Since the exponent contains <b>n²</b>,
higher-order modes decay much faster than the low-order modes.
Consequently, the temperature profile becomes progressively smoother
and is increasingly dominated by the lowest spatial modes.
</div>

<div style="margin-bottom:6px;">
The stem plot on the right shows this behavior directly. Each vertical
line corresponds to one discrete spatial mode n, and its height represents
the magnitude of its current modal contribution.
</div>

<div>
Because the boundary conditions are <b>u(0,t)=u(L,t)=0</b>, all Fourier
modes eventually decay to zero. Thus, in the limit <b>t → ∞</b>,
the complete temperature distribution approaches the horizontal
equilibrium line <b>u(x,t)=0</b>.
</div>

</div>
""")

# ============================================================
# UPDATE FUNCTION
#
# Figures are not recreated.
# Only existing curves, points and stem segments are updated.
# ============================================================

def update_notebook(change=None):

    global coefficients

    t = (
        time_slider.value
    )

    sigma = (
        sigma_slider.value
    )

    N = (
        modes_slider.value
    )

    # --------------------------------------------------------
    # Initial condition
    # --------------------------------------------------------

    f_values = initial_function(
        initial_selector.value
    )

    coefficients = calculate_coefficients(
        f_values
    )

    # --------------------------------------------------------
    # Heat-equation solution
    # --------------------------------------------------------

    u = build_solution(
        t,
        sigma,
        N
    )

    initial_line.set_ydata(
        f_values
    )

    solution_line.set_ydata(
        u
    )

    # --------------------------------------------------------
    # Modal amplitudes
    # --------------------------------------------------------

    modal_values = np.zeros(
        MAX_MODES
    )

    n = mode_numbers[
        :N
    ]

    modal_values[:N] = (
        np.abs(
            coefficients[:N]
        )
        *
        np.exp(
            -sigma
            *
            (n*np.pi/L)**2
            *
            t
        )
    )

    # --------------------------------------------------------
    # Stem markers
    # --------------------------------------------------------

    modal_points.set_data(
        mode_numbers,
        modal_values
    )

    # --------------------------------------------------------
    # Stem vertical lines
    # --------------------------------------------------------

    segments = []

    for index, mode in enumerate(
        mode_numbers
    ):

        segments.append(
            [
                (
                    mode,
                    0.0
                ),
                (
                    mode,
                    modal_values[index]
                )
            ]
        )

    stem_lines.set_segments(
        segments
    )

    # --------------------------------------------------------
    # Show only retained modes
    # --------------------------------------------------------

    ax_modes.set_xlim(
        0.5,
        N + 0.5
    )

    # --------------------------------------------------------
    # Dynamic vertical range
    # --------------------------------------------------------

    max_modal = np.max(
        modal_values[:N]
    )

    if max_modal > 0.0:

        ax_modes.set_ylim(
            0.0,
            1.10 * max_modal
        )

    else:

        ax_modes.set_ylim(
            0.0,
            1.0
        )

    # --------------------------------------------------------
    # Slider values
    # --------------------------------------------------------

    time_value.value = (
        f'<div class="heat-value">{t:.3f}</div>'
    )

    sigma_value.value = (
        f'<div class="heat-value">{sigma:.2f}</div>'
    )

    modes_value.value = (
        f'<div class="heat-value">{N}</div>'
    )

    # --------------------------------------------------------
    # Current solution information
    # --------------------------------------------------------

    info_math.value = (
        r'\('
        r'N='
        +
        str(N)
        +
        r',\;'
        r't='
        +
        f'{t:.4f}'
        +
        r',\;'
        r'\sigma='
        +
        f'{sigma:.3f}'
        +
        r'\)'
    )

    # --------------------------------------------------------
    # Redraw only
    # --------------------------------------------------------

    fig_profile.canvas.draw_idle()
    fig_modes.canvas.draw_idle()

# ============================================================
# CONNECT CONTROLS
# ============================================================

initial_selector.observe(
    update_notebook,
    names='value'
)

time_slider.observe(
    update_notebook,
    names='value'
)

sigma_slider.observe(
    update_notebook,
    names='value'
)

modes_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        relations,
        top_row,
        figures_row,
        conclusion_panel
    ],
    layout=Layout(
        width='1180px',
        gap='10px'
    )
)

display(
    main_layout
)